# SPX Gamma / Steven / 15 分钟判断模型审计

技术审计窗口为 2026-07-03–2026-07-23。报告方向结果是重叠的观察样本，不能当作独立交易或净收益。

In [1]:
from pathlib import Path
import sys

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/spx_spark").is_dir()
)
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from build_gamma_decision_model_audit import collect_analysis  # noqa: E402

analysis = collect_analysis()
analysis["report_counts"]

{'total': 521,
 'RTH': 129,
 'GTH': 392,
 'zero_gamma_transition': 397,
 'negative_gamma_acceleration': 57,
 'positive_gamma_pin': 1,
 'mixed_gamma': 1,
 'gamma_missing': 65,
 'score_observations': 293}

## Gamma classification quality

Gamma state is a call-positive / put-negative OI-volume structure proxy. It is not a dealer-position estimate.

In [2]:
analysis["gamma_distribution"], analysis["zero_gamma_distance"], analysis["current_quality"]

([{'session': 'RTH',
   'gamma_state': 'ZG',
   'count': 125,
   'share': 0.9689922480620154},
  {'session': 'RTH',
   'gamma_state': 'Neg',
   'count': 3,
   'share': 0.023255813953488372},
  {'session': 'RTH', 'gamma_state': 'Pos', 'count': 0, 'share': 0.0},
  {'session': 'RTH',
   'gamma_state': 'Mix',
   'count': 1,
   'share': 0.007751937984496124},
  {'session': 'RTH', 'gamma_state': 'Missing', 'count': 0, 'share': 0.0},
  {'session': 'GTH',
   'gamma_state': 'ZG',
   'count': 272,
   'share': 0.6938775510204082},
  {'session': 'GTH',
   'gamma_state': 'Neg',
   'count': 54,
   'share': 0.1377551020408163},
  {'session': 'GTH',
   'gamma_state': 'Pos',
   'count': 1,
   'share': 0.002551020408163265},
  {'session': 'GTH', 'gamma_state': 'Mix', 'count': 0, 'share': 0.0},
  {'session': 'GTH',
   'gamma_state': 'Missing',
   'count': 65,
   'share': 0.16581632653061223}],
 [{'session': 'ALL',
   'n': 364,
   'median_distance_points': 8.949999999999818,
   'within_0_10_pct': 0.425824

## Report-score calibration

The score outcome joins each report to a same-session report at the requested horizon ±3 minutes.

In [3]:
analysis["rth_score_60m"], analysis["rth_mode_60m"]

([{'session': 'RTH',
   'score_bucket': '0–44',
   'horizon': '60m',
   'n': 9,
   'hit_rate': 0.5555555555555556,
   'mean_signed_es_points': 0.7444444444444243,
   'median_signed_es_points': 6.699999999999818},
  {'session': 'RTH',
   'score_bucket': '45–64',
   'horizon': '60m',
   'n': 9,
   'hit_rate': 0.7777777777777778,
   'mean_signed_es_points': 6.255555555555576,
   'median_signed_es_points': 9.199999999999818},
  {'session': 'RTH',
   'score_bucket': '65+',
   'horizon': '60m',
   'n': 32,
   'hit_rate': 0.5,
   'mean_signed_es_points': -4.381249999999966,
   'median_signed_es_points': 1.099999999999909}],
 [{'session': 'RTH',
   'mode': 'trending',
   'horizon': '60m',
   'n': 39,
   'hit_rate': 0.5128205128205128,
   'mean_signed_es_points': -3.320512820512774},
  {'session': 'RTH',
   'mode': 'transition',
   'horizon': '60m',
   'n': 21,
   'hit_rate': 0.6666666666666666,
   'mean_signed_es_points': 3.7857142857142856}])

## Steven separation and dead-end audit

Steven is observe-only and is not an input to the 15-minute report guidance.

In [4]:
analysis["steven"], analysis["steven_states"][:10]

({'events': 5324,
  'map_revisions': 3725,
  'state_transitions': 1586,
  'setup_confirmed': 0,
  'low_confidence': 5324,
  'missing_bars_1m': 5323,
  'missing_es_volume': 5323,
  'missing_hl_volume': 5323,
  'empty_bar_files': 13},
 [{'machine_state': 'REGIME_UNKNOWN', 'count': 2583},
  {'machine_state': 'BEARISH_BREAK_WATCH', 'count': 1262},
  {'machine_state': 'OBSERVE_ONLY', 'count': 611},
  {'machine_state': 'DATA_INVALID', 'count': 383},
  {'machine_state': 'BULLISH_DIP_WATCH', 'count': 351},
  {'machine_state': 'RANGE_PIN_WATCH', 'count': 105},
  {'machine_state': 'EVENT_WAIT', 'count': 29}])

## RTH action funnel and report-cadence mismatch

Rows are persisted intent-signature changes, so both record counts and unique event counts are retained.

In [5]:
analysis["trade_intent"], analysis["trade_intent_rows"]

({'records': 18822,
  'observing': 18706,
  'blocked': 114,
  'trade_ready': 2,
  'nonobserving_unique_events': 10,
  'trade_ready_unique_events': 2,
  'ready_rows': [{'event_id': 'level:84c4346b55f19eb00429c5a6',
    'evaluated_at': '2026-07-15T15:50:51.090773+00:00',
    'evaluated_at_et': '2026-07-15T11:50:51.090773-04:00',
    'valid_until': None},
   {'event_id': 'level:a2799ffe7e2fe7a8516e59e5',
    'evaluated_at': '2026-07-14T14:33:52.983728+00:00',
    'evaluated_at_et': '2026-07-14T10:33:52.983728-04:00',
    'valid_until': None}]},
 [{'status': 'observing', 'signature_records': 18706, 'unique_events': 167},
  {'status': 'blocked', 'signature_records': 114, 'unique_events': 9},
  {'status': 'trade_ready', 'signature_records': 2, 'unique_events': 2}])

In [6]:
assert analysis["report_counts"]["total"] == 521
assert analysis["report_counts"]["RTH"] == 129
assert analysis["report_counts"]["zero_gamma_transition"] == 397
assert analysis["steven"]["setup_confirmed"] == 0
assert analysis["steven"]["events"] == 5324
assert analysis["trade_intent"]["records"] == 18822
assert analysis["trade_intent"]["trade_ready"] == 2
assert analysis["trade_intent"]["trade_ready_unique_events"] == 2
assert analysis["current_quality"]["low_coverage_ready_frames"] > 0
print("VALIDATED: report/Gamma population, low-coverage classification, Steven dead-end, and RTH action funnel.")

VALIDATED: report/Gamma population, low-coverage classification, Steven dead-end, and RTH action funnel.
